In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlssmartdada2";

In [0]:
%sql
create widget text catalogo default "catalog_smartdata_final";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
%sql
DROP CATALOG IF EXISTS ${catalogo} CASCADE;

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ${catalogo}
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
%sql
DROP SCHEMA IF EXISTS ${catalogo}.raw;
DROP SCHEMA IF EXISTS ${catalogo}.bronze;
DROP SCHEMA IF EXISTS ${catalogo}.silver;
DROP SCHEMA IF EXISTS ${catalogo}.golden;

In [0]:
catalogo = dbutils.widgets.get("catalogo")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.raw;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.silver;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.golden;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.exploratory;""")

### Tablas para raw - Especificaciones vehículos eléctricos

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.raw.ev_specs_2025 (
  brand STRING,
  model STRING,
  top_speed_kmh INT,
  battery_capacity_kwh DOUBLE,
  battery_type STRING,
  number_of_cells INT,
  torque_nm INT,
  efficiency_wh_per_km DOUBLE,
  range_km INT,
  acceleration_0_100_s DOUBLE,
  fast_charging_power_kw_dc DOUBLE,
  fast_charge_port STRING,
  towing_capacity_kg STRING,
  cargo_volume_l STRING,
  seats INT,
  drivetrain STRING,
  segment STRING,
  length_mm INT,
  width_mm INT,
  height_mm INT,
  car_body_type STRING,
  source_url STRING
 )
USING DELTA
LOCATION 'abfss://raw@{storageName}.dfs.core.windows.net/{catalogo}/ev_specs_2025';
""")

### Tablas para raw - EV Population


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.raw.ev_population_data (
  vin_1_10 STRING,
  county STRING,
  city STRING,
  state STRING,
  postal_code STRING,
  model_year INT,
  make STRING,
  model STRING,
  electric_vehicle_type STRING,
  cafv_eligibility STRING,
  electric_range INT,
  base_msrp INT,
  legislative_district INT,
  dol_vehicle_id BIGINT,
  vehicle_location STRING,
  electric_utility STRING,
  census_tract_2020 BIGINT
 )
USING DELTA
LOCATION 'abfss://raw@{storageName}.dfs.core.windows.net/{catalogo}/ev_population_data';
""")

### Tablas BRONZE (datos crudos estandarizados)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.bronze.ev_specs_2025_bronze (
  brand STRING,
  model STRING,
  top_speed_kmh INT,
  battery_capacity_kwh DOUBLE,
  battery_type STRING,
  number_of_cells INT,
  torque_nm INT,
  efficiency_wh_per_km DOUBLE,
  range_km INT,
  acceleration_0_100_s DOUBLE,
  fast_charging_power_kw_dc DOUBLE,
  fast_charge_port STRING,
  towing_capacity_kg STRING,
  cargo_volume_l STRING,
  seats INT,
  drivetrain STRING,
  segment STRING,
  length_mm INT,
  width_mm INT,
  height_mm INT,
  car_body_type STRING,
  source_url STRING,
  source_file STRING,
  ingestion_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/{catalogo}/ev_specs_2025_bronze';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.bronze.ev_population_bronze (
  vin_1_10 STRING,
  county STRING,
  city STRING,
  state STRING,
  postal_code STRING,
  model_year INT,
  make STRING,
  model STRING,
  electric_vehicle_type STRING,
  cafv_eligibility STRING,
  electric_range INT,
  base_msrp INT,
  legislative_district INT,
  dol_vehicle_id BIGINT,
  vehicle_location STRING,
  electric_utility STRING,
  census_tract_2020 BIGINT,
  source_file STRING,
  ingestion_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/{catalogo}/ev_population_bronze';
""")

### Tablas SILVER (modelo integrado y listo para análisis)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.silver.ev_model_specs (
  brand STRING,
  model STRING,
  join_make STRING,
  join_model STRING,
  battery_capacity_kwh DOUBLE,
  range_km INT,
  efficiency_wh_per_km DOUBLE,
  fast_charging_power_kw_dc DOUBLE,
  seats INT,
  drivetrain STRING,
  segment STRING,
  car_body_type STRING,
  source_url STRING,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/{catalogo}/ev_model_specs';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.silver.ev_population_clean (
  vin_1_10 STRING,
  state STRING,
  county STRING,
  city STRING,
  postal_code STRING,
  model_year INT,
  make STRING,
  model STRING,
  join_make STRING,
  join_model STRING,
  electric_vehicle_type STRING,
  cafv_eligibility STRING,
  electric_range INT,
  electric_range_km DOUBLE,
  base_msrp INT,
  has_valid_msrp BOOLEAN,
  dol_vehicle_id BIGINT,
  vehicle_location STRING,
  electric_utility STRING,
  census_tract_2020 BIGINT,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/{catalogo}/ev_population_clean';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.silver.ev_vehicle_enriched (
  vin_1_10 STRING,
  state STRING,
  county STRING,
  city STRING,
  model_year INT,
  make STRING,
  model STRING,
  join_make STRING,
  join_model STRING,
  electric_vehicle_type STRING,
  cafv_eligibility STRING,
  registered_electric_range INT,
  registered_electric_range_km DOUBLE,
  base_msrp INT,
  has_valid_msrp BOOLEAN,
  specs_brand STRING,
  specs_model STRING,
  specs_battery_capacity_kwh DOUBLE,
  specs_range_km DOUBLE,
  specs_efficiency_wh_per_km DOUBLE,
  specs_fast_charging_power_kw_dc DOUBLE,
  specs_drivetrain STRING,
  specs_segment STRING,
  specs_car_body_type STRING,
  has_exact_match BOOLEAN,
  has_prefix_match BOOLEAN,
  has_specs_match BOOLEAN,
  specs_variant_count BIGINT,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/{catalogo}/ev_vehicle_enriched';
""")


### Tablas GOLDEN (métricas de negocio para consumo)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.golden.golden_ev_model_popularity (
  make STRING,
  model STRING,
  vehicles_registered BIGINT,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/{catalogo}/golden_ev_model_popularity';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.golden.golden_make_price_range (
  make STRING,
  avg_price DOUBLE,
  avg_range DOUBLE,
  total_vehicles BIGINT,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/{catalogo}/golden_make_price_range';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.golden.golden_state_distribution (
  state STRING,
  total_ev BIGINT,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/{catalogo}/golden_state_distribution';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.golden.golden_range_vs_registrations (
  make STRING,
  model STRING,
  specs_range_km INT,
  vehicles_registered BIGINT,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/{catalogo}/golden_range_vs_registrations';
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.golden.golden_make_range_price_ratio (
  make STRING,
  avg_specs_range_km DOUBLE,
  avg_price DOUBLE,
  range_per_price_ratio DOUBLE,
  updated_ts TIMESTAMP
 )
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/{catalogo}/golden_make_range_price_ratio';
""")

In [0]:
schemas = ["raw", "bronze", "silver", "golden"]
tables_by_schema = []

for schema in schemas:
    df = spark.sql(f"SHOW TABLES IN {catalogo}.{schema}")
    if "database" in df.columns:
        df = df.selectExpr(
            f"'{schema}' AS schema_name",
            "database AS schema_db",
            "tableName",
            "isTemporary"
        )
    else:
        df = df.selectExpr(
            f"'{schema}' AS schema_name",
            "namespace AS schema_db",
            "tableName",
            "isTemporary"
        )
    tables_by_schema.append(df)

result_df = tables_by_schema[0]
for i in range(1, len(tables_by_schema)):
    result_df = result_df.unionByName(tables_by_schema[i])

display(result_df.orderBy("schema_name", "tableName"))